## Задание 1. Линейная регрессия вручную

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

data = np.genfromtxt('data.csv', delimiter=',', skip_header=1)
X = data[:, :-1]
y = data[:, -1]
print(data[:5])

fig, axes = plt.subplots(1, X.shape[1], figsize=(10, 4))
for i in range(X.shape[1]):
    axes[i].scatter(X[:, i], y)
    axes[i].set_xlabel(f'X{i + 1}')
    axes[i].set_ylabel('y')
plt.tight_layout()
plt.show()

X_with_const = np.column_stack([np.ones(len(X)), X])
coefficients = np.linalg.lstsq(X_with_const, y, rcond=None)[0]
manual_predictions = X_with_const @ coefficients
print('Коэффициенты:', coefficients)
print('MSE:', mean_squared_error(y, manual_predictions))

[[ 3.74540119  3.92587981 24.82642976]
 [ 9.50714306  0.99836891 35.6895353 ]
 [ 7.31993942  2.57117219 31.98651436]
 [ 5.98658484  2.96207284 28.58279652]
 [ 1.5601864   0.23225206  8.66654135]]
Коэффициенты: [5.09151466 2.983307   1.94436153]
MSE: 0.6360953254996485


/tmp/ipykernel_5178/1761673677.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Задание 2. Регрессия statsmodels и прогноз

In [2]:
test = np.genfromtxt('data_test.csv', delimiter=',', skip_header=1)
X_test = test[:, :-1]
y_test = test[:, -1]
ols = sm.OLS(y, sm.add_constant(X)).fit()
ols_predictions = ols.predict(sm.add_constant(X_test))
print(ols.summary())
print(pd.DataFrame({'Факт': y_test, 'Прогноз': ols_predictions}))
print('R2:', r2_score(y_test, ols_predictions))
print('MSE:', mean_squared_error(y_test, ols_predictions))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                     1273.
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           1.82e-23
Time:                        16:56:02   Log-Likelihood:                -29.818
No. Observations:                  25   AIC:                             65.64
Df Residuals:                      22   BIC:                             69.29
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.0915      0.427     11.919      0.0

## Задание 3. Градиентный спуск

In [3]:
x = X[:, 0]
x_scaled = (x - x.mean()) / x.std()
a = 0.0
b = y.mean()
learning_rate = 0.01
for i in range(5000):
    prediction = a * x_scaled + b
    a -= learning_rate * np.mean(2 * (prediction - y) * x_scaled)
    b -= learning_rate * np.mean(2 * (prediction - y))
gradient_predictions = a * x_scaled + b
print('Коэффициент:', a)
print('Свободный член:', b)
print('MSE:', mean_squared_error(y, gradient_predictions))
order = np.argsort(x)
plt.scatter(x, y)
plt.plot(x[order], gradient_predictions[order], color='red')
plt.xlabel('X1')
plt.ylabel('y')
plt.show()

Коэффициент: 8.10066990583377
Свободный член: 22.626907029837547
MSE: 8.653676337653948


/tmp/ipykernel_5178/3153468990.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Задание 4. Регрессия Scikit-learn

In [4]:
sklearn_model = LinearRegression()
sklearn_model.fit(X, y)
print('Statsmodels:', ols.params)
print('Scikit-learn:', np.r_[sklearn_model.intercept_, sklearn_model.coef_])
print('Совпадение:', np.allclose(ols.params, np.r_[sklearn_model.intercept_, sklearn_model.coef_]))

Statsmodels: [5.09151466 2.983307   1.94436153]
Scikit-learn: [5.09151466 2.983307   1.94436153]
Совпадение: True


## Задание 5. Коэффициент детерминации

In [5]:
sklearn_predictions = sklearn_model.predict(X_test)
print('R2 на обучающих данных:', r2_score(y, sklearn_model.predict(X)))
print('R2 на тестовых данных:', r2_score(y_test, sklearn_predictions))

R2 на обучающих данных: 0.9914358888325637
R2 на тестовых данных: 0.9735104772203199


## Самостоятельная работа. Недвижимость

In [6]:
train = pd.read_csv('Nedvig.csv', sep=';', encoding='cp1251').dropna()
test = pd.read_csv('Nedvig_test.csv', sep=';', encoding='cp1251')
X_house = pd.get_dummies(train.iloc[:, 1:-1], drop_first=True, dtype=float)
y_house = train.iloc[:, -1]
X_house_test = pd.get_dummies(test.iloc[:, 1:-1], drop_first=True, dtype=float)
X_house_test = X_house_test.reindex(columns=X_house.columns, fill_value=0)
y_house_test = test.iloc[:, -1]

plt.scatter(train['Общая'], y_house)
plt.xlabel('Общая площадь')
plt.ylabel('Цена')
plt.show()

house_ols = sm.OLS(y_house, sm.add_constant(X_house)).fit()
house_ols_predictions = house_ols.predict(sm.add_constant(X_house_test, has_constant='add'))
house_model = LinearRegression()
house_model.fit(X_house, y_house)
house_predictions = house_model.predict(X_house_test)
print(house_ols.summary())
print(pd.DataFrame({'Факт': y_house_test, 'Statsmodels': house_ols_predictions, 'Scikit-learn': house_predictions}))
print('R2 Statsmodels:', r2_score(y_house_test, house_ols_predictions))
print('R2 Scikit-learn:', r2_score(y_house_test, house_predictions))

                            OLS Regression Results                            
Dep. Variable:        Цена, тыс. руб.   R-squared:                       0.287
Model:                            OLS   Adj. R-squared:                  0.271
Method:                 Least Squares   F-statistic:                     18.55
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           5.75e-81
Time:                        16:56:03   Log-Likelihood:                -12017.
No. Observations:                1414   AIC:                         2.410e+04
Df Residuals:                    1383   BIC:                         2.426e+04
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

/tmp/ipykernel_5178/969093652.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_5178/969093652.py:14: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  house_ols = sm.OLS(y_house, sm.add_constant(X_house)).fit()


## Ответы на контрольные вопросы

1. **Задача регрессии** — прогнозирование числового значения по одному или нескольким признакам.
2. **Линейная регрессия** описывает зависимость результата от признаков линейной функцией.
3. **Множественная линейная регрессия** использует несколько независимых переменных одновременно.
4. **Метод наименьших квадратов** выбирает коэффициенты, при которых сумма квадратов ошибок прогноза минимальна.
5. Основные ограничения МНК: линейность связи, независимость ошибок, постоянство их дисперсии, отсутствие сильной мультиколлинеарности и существенных выбросов. Для статистических выводов также желательна нормальность остатков.